# 03 — Adaptação binária → triagem em 3 níveis

**Projeto:** Medical Triage MLOps  
**Modelo-base:** `Yuvrajxms09/biobert-triage-classifier`

## Objetivo

O modelo pré-treinado produz duas probabilidades:

- `urgent`
- `non-urgent`

Para adaptar essa saída ao objetivo acadêmico do projeto, usamos uma **regra de decisão com zona de incerteza**:

```text
alta confiança em urgent      → urgente
baixa confiança / ambiguidade → atenção
alta confiança em non-urgent  → normal
```

A classe **atenção** não é uma classe aprendida diretamente pelo BioBERT. Ela representa uma **zona de incerteza operacional** em que o modelo não demonstra confiança suficiente para classificar o caso como claramente urgente ou claramente não urgente.

> Esta abordagem é uma heurística de pseudo-rotulagem para fins acadêmicos. Ela não corresponde a uma escala clínica validada.

## 1. Bibliotecas e configurações

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

CONFIDENCE_THRESHOLD = 0.70

COLORS = {
    "normal": "#16A085",
    "atenção": "#F4B942",
    "urgente": "#D63031",
}

## 2. Carregamento do piloto

Este notebook parte do arquivo:

```text
data/processed/triage_pilot_biobert.csv
```

Ele deve conter:

- `medical_abstract`
- `urgent_score`
- `nonurgent_score`
- `confidence`

In [ ]:
def locate_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, current.parent, current.parent.parent]:
        expected = candidate / "data" / "processed" / "triage_pilot_biobert.csv"
        if expected.exists():
            return candidate

    raise FileNotFoundError(
        "Não encontrei data/processed/triage_pilot_biobert.csv. "
        "Execute primeiro o notebook 02_pilot_pseudolabel_biobert.ipynb."
    )


PROJECT_ROOT = locate_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

pilot_path = PROCESSED_DIR / "triage_pilot_biobert.csv"
pilot = pd.read_csv(pilot_path)

required_columns = {
    "medical_abstract",
    "urgent_score",
    "nonurgent_score",
    "confidence",
}

missing = required_columns - set(pilot.columns)

if missing:
    raise ValueError(f"Colunas ausentes no arquivo piloto: {sorted(missing)}")

print(f"Registros carregados: {len(pilot)}")
display(pilot.head())

## 3. Regra de adaptação para três classes

Com `CONFIDENCE_THRESHOLD = 0.70`:

- **urgente**: `urgent_score >= 0.70`
- **normal**: `nonurgent_score >= 0.70`
- **atenção**: nenhum dos lados alcança 0.70

Como as probabilidades somam aproximadamente 1, a zona de atenção fica aproximadamente entre `0.30` e `0.70` no `urgent_score`.

In [ ]:
def map_to_three_levels(
    urgent_score: float,
    nonurgent_score: float,
    threshold: float = CONFIDENCE_THRESHOLD,
) -> str:
    if urgent_score >= threshold:
        return "urgente"

    if nonurgent_score >= threshold:
        return "normal"

    return "atenção"


pilot["triage_level"] = pilot.apply(
    lambda row: map_to_three_levels(
        urgent_score=row["urgent_score"],
        nonurgent_score=row["nonurgent_score"],
    ),
    axis=1,
)

display(
    pilot[
        [
            "medical_abstract",
            "urgent_score",
            "nonurgent_score",
            "confidence",
            "triage_level",
        ]
    ].head(10)
)

## 4. Distribuição das três classes

In [ ]:
order = ["normal", "atenção", "urgente"]

distribution = (
    pilot["triage_level"]
    .value_counts()
    .reindex(order, fill_value=0)
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

distribution["percentual"] = distribution["quantidade"] / len(pilot)

display(distribution)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    distribution["classe"],
    distribution["quantidade"],
    color=[COLORS[label] for label in distribution["classe"]],
)

ax.bar_label(
    bars,
    labels=[
        f"{count} ({pct:.1%})"
        for count, pct in zip(
            distribution["quantidade"],
            distribution["percentual"],
        )
    ],
    padding=4,
)

ax.set_title(
    f"Triagem em três níveis — threshold = {CONFIDENCE_THRESHOLD:.2f}"
)
ax.set_ylabel("Quantidade de casos")

plt.tight_layout()
plt.show()

## 5. Visualização da zona de decisão

O gráfico mostra o `urgent_score`:

- valores altos → urgente;
- valores baixos → normal;
- região central → atenção.

In [ ]:
sorted_pilot = pilot.sort_values("urgent_score").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 5))

for triage_class in order:
    mask = sorted_pilot["triage_level"] == triage_class
    ax.scatter(
        sorted_pilot.index[mask],
        sorted_pilot.loc[mask, "urgent_score"],
        label=triage_class,
        color=COLORS[triage_class],
        s=45,
        alpha=0.85,
    )

lower_boundary = 1 - CONFIDENCE_THRESHOLD
upper_boundary = CONFIDENCE_THRESHOLD

ax.axhline(
    lower_boundary,
    color="#636E72",
    linestyle="--",
    linewidth=1.5,
)
ax.axhline(
    upper_boundary,
    color="#636E72",
    linestyle="--",
    linewidth=1.5,
)

ax.axhspan(
    lower_boundary,
    upper_boundary,
    color="#F4B942",
    alpha=0.08,
)

ax.set_title("Zona de decisão baseada no score de urgência")
ax.set_xlabel("Registros ordenados pelo score")
ax.set_ylabel("urgent_score")
ax.set_ylim(0, 1)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 6. Sensibilidade ao threshold

Quanto maior o threshold, maior tende a ser a classe `atenção`.

In [ ]:
thresholds = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85]

threshold_results = []

for threshold in thresholds:
    labels_for_threshold = pilot.apply(
        lambda row: map_to_three_levels(
            urgent_score=row["urgent_score"],
            nonurgent_score=row["nonurgent_score"],
            threshold=threshold,
        ),
        axis=1,
    )

    counts = labels_for_threshold.value_counts()

    threshold_results.append(
        {
            "threshold": threshold,
            "normal": counts.get("normal", 0),
            "atenção": counts.get("atenção", 0),
            "urgente": counts.get("urgente", 0),
        }
    )

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for triage_class in order:
    ax.plot(
        threshold_df["threshold"],
        threshold_df[triage_class],
        marker="o",
        linewidth=2,
        label=triage_class,
        color=COLORS[triage_class],
    )

ax.set_title("Impacto do threshold na distribuição das classes")
ax.set_xlabel("Threshold mínimo de confiança")
ax.set_ylabel("Quantidade de casos")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

### Interpretação

- threshold baixo: mais decisões diretas como `normal` ou `urgente`;
- threshold alto: mais registros caem em `atenção`.

O valor `0.70` é usado apenas como ponto inicial de projeto. Ele deve ser documentado e pode ser ajustado após inspeção dos resultados.

## 7. Exemplos por classe

In [ ]:
for triage_class in order:
    display(Markdown(f"### {triage_class.upper()}"))

    examples = (
        pilot.loc[pilot["triage_level"] == triage_class]
        .sort_values("confidence", ascending=False)
        [
            [
                "medical_abstract",
                "urgent_score",
                "nonurgent_score",
                "confidence",
            ]
        ]
        .head(5)
    )

    display(examples)

## 8. Casos classificados como atenção

Esses casos não foram classificados como `atenção` porque o BioBERT aprendeu essa classe.  
Eles estão nessa categoria porque ficaram dentro da **zona de incerteza definida pelo projeto**.

In [ ]:
attention_cases = (
    pilot.loc[pilot["triage_level"] == "atenção"]
    .assign(distance_from_middle=lambda df: (df["urgent_score"] - 0.5).abs())
    .sort_values("distance_from_middle")
    [
        [
            "medical_abstract",
            "urgent_score",
            "nonurgent_score",
            "confidence",
        ]
    ]
)

display(attention_cases.head(15))

## 9. Exportação

In [ ]:
output_file = PROCESSED_DIR / "triage_pilot_three_levels.csv"

pilot.to_csv(output_file, index=False)

print(f"Arquivo salvo em: {output_file}")

## 10. Conclusão metodológica

A estratégia utilizada é uma **adaptação de um classificador binário com zona de incerteza**:

```text
urgent com alta confiança      → urgente
resultado incerto              → atenção
non-urgent com alta confiança  → normal
```

No trabalho, a classe `atenção` deve ser apresentada como:

> casos em que o modelo pré-treinado não demonstrou confiança suficiente para classificá-los como claramente urgentes ou não urgentes, sendo encaminhados para uma categoria intermediária de atenção.

Ela não deve ser descrita como uma classe clínica originalmente aprendida pelo BioBERT.